In [1]:
import logging
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
#from sklearn.metrics import plot_roc_curve
from sklearn.model_selection import ParameterGrid
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
import copy
from sklearn.model_selection import StratifiedKFold, train_test_split
import os
import pickle
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef,
    confusion_matrix, log_loss, roc_curve
)
import sys
sys.path.append("../src")

from act_dl import TorchMLPClassifier

/Users/idahyeon/miniforge3/envs/adrb2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# =========================
# 1. config
# =========================
INPATH = "../data"
OUTPATH = "../data/modelago"
os.makedirs(OUTPATH, exist_ok=True)

DATA_FILE = os.path.join(INPATH, "forModelAgo.csv")
TARGET_COL = "Agonist"

extra_features = ['nBase','ATSC4c','ATSC5c','ATSC5d','ATSC4i','AATSC0v','AATSC1pe','AATSC1p',
           'MATS1se','MATS1pe','MATS1are','MATS1p','MATS1i','BalabanJ','RPCG','C4SP3',
           'Xch-3d','Xch-4d','Xc-4dv','NsssB','NssssB','NaaNH','NdsN','NsOH','NdS','NdssS',
           'SsssB','SssssB','SaaNH','SdsN','SsOH','SdssS','IC2','IC3','MIC2','n7Ring','n7HRing',
           'n5AHRing','n7AHRing','nFRing','n8FRing','n10FRing','nG12FRing','nFHRing','n10FHRing',
           'nG12FHRing','nG12FaRing','nFaHRing','n10FaHRing','nG12FaHRing','n10FARing','n8FAHRing','n10FAHRing','JGI4']

# ecfp6 기준
X_COLS = [f"X{i}" for i in range(1, 1025)] + extra_features

PRED_THRESHOLD = 0.7
RANDOM_STATE = 42

# =====================================
# 2. 고정 하이퍼파라미터
# =====================================
MODEL_PARAMS = {
    "xgb": {
        "colsample_bytree": 0.6,
        "gamma": 20,
        "reg_lambda": 0.09,
        "learning_rate": 0.1,
        "max_depth": 4,
        "min_child_weight": 9,
        "n_estimators": 1000,
        "subsample": 0.9,
        "use_label_encoder": False,
        "verbosity": 0,
        "random_state": 42,
        "eval_metric": "logloss"
    },
    "rf": {
        "max_depth": 20,
        "min_samples_leaf": 1,
        "min_samples_split": 2,
        "random_state": 42
    },
    "dl": {
        "batch_size": 64,
        "dropout_rate": 0.3138483452528049,
        "epochs": 46,
        "hidden_size": 128,
        "learning_rate": 0.0017856801079863586,
        "weight_decay":  0.0006483710967545515
    }
}





In [ ]:
# =====================================
# 3. evaluate function
# =====================================
def evaluate_binary(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1]
    }

In [ ]:
# =====================================
# 4. model
# =====================================
def build_model(mode):
    if mode == "xgb":
        return xgb.XGBClassifier(**MODEL_PARAMS["xgb"])

    elif mode == "rf":
        return RandomForestClassifier(**MODEL_PARAMS["rf"])

    elif mode == "dl":
        return TorchMLPClassifier(
            hidden_size=MODEL_PARAMS["dl"]["hidden_size"],
            dropout_rate=MODEL_PARAMS["dl"]["dropout_rate"],
            learning_rate=MODEL_PARAMS["dl"]["learning_rate"],
            weight_decay=MODEL_PARAMS["dl"]["weight_decay"],
            epochs=MODEL_PARAMS["dl"]["epochs"],
            batch_size=MODEL_PARAMS["dl"]["batch_size"],
            random_state=42,
            verbose=0
        )

    else:
        raise ValueError(f"Unsupported mode: {mode}")

In [9]:
from tqdm.auto import tqdm

def run_model_cv(mode, df, X_COLS, TARGET_COL, outpath, pred_threshold=0.7, random_state=42):
    X = df[X_COLS]
    y = df[TARGET_COL]

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_state)
    results = []

    splits = list(skf.split(X, y))

    for fold_idx, (trainval_idx, test_idx) in enumerate(
        tqdm(splits, desc=f"{mode.upper()} CV folds", total=len(splits)),
        start=1
    ):
        trainval_df = df.iloc[trainval_idx].copy()
        test_df = df.iloc[test_idx].copy()

        train_df, val_df = train_test_split(
            trainval_df,
            test_size=1/9,
            stratify=trainval_df[TARGET_COL],
            random_state=random_state + fold_idx
        )

        x_train = train_df[X_COLS]
        y_train = train_df[TARGET_COL]

        x_val = val_df[X_COLS]
        y_val = val_df[TARGET_COL]

        x_test = test_df[X_COLS]
        y_test = test_df[TARGET_COL]

        model = build_model(mode)
        model.fit(x_train, y_train)

        train_prob = model.predict_proba(x_train)[:, 1]
        train_pred = (train_prob >= pred_threshold).astype(int)
        train_metrics = evaluate_binary(y_train, train_pred, train_prob)

        val_prob = model.predict_proba(x_val)[:, 1]
        val_pred = (val_prob >= pred_threshold).astype(int)
        val_metrics = evaluate_binary(y_val, val_pred, val_prob)

        test_prob = model.predict_proba(x_test)[:, 1]
        test_pred = (test_prob >= pred_threshold).astype(int)
        test_metrics = evaluate_binary(y_test, test_pred, test_prob)

        print(f"\n===== {mode.upper()} Fold {fold_idx} =====")
        print(f"Train / Val / Test = {len(train_df)} / {len(val_df)} / {len(test_df)}")
        print(f"Test Recall: {test_metrics['recall']:.4f}")
        print(f"Test MCC:    {test_metrics['mcc']:.4f}")
        print(f"Test AUC:    {test_metrics['roc_auc']:.4f}")

        results.append({
            "fold": fold_idx,
            "tr_accuracy": train_metrics["accuracy"],
            "tr_precision": train_metrics["precision"],
            "tr_recall": train_metrics["recall"],
            "tr_f1": train_metrics["f1"],
            "tr_roc_auc": train_metrics["roc_auc"],
            "tr_mcc": train_metrics["mcc"],
            "v_accuracy": val_metrics["accuracy"],
            "v_precision": val_metrics["precision"],
            "v_recall": val_metrics["recall"],
            "v_f1": val_metrics["f1"],
            "v_roc_auc": val_metrics["roc_auc"],
            "v_mcc": val_metrics["mcc"],
            "te_accuracy": test_metrics["accuracy"],
            "te_precision": test_metrics["precision"],
            "te_recall": test_metrics["recall"],
            "te_f1": test_metrics["f1"],
            "te_roc_auc": test_metrics["roc_auc"],
            "te_mcc": test_metrics["mcc"],
            "tn": test_metrics["tn"],
            "fp": test_metrics["fp"],
            "fn": test_metrics["fn"],
            "tp": test_metrics["tp"],
        })

        with open(os.path.join(f"{outpath}/{mode}", f"{mode}_fold{fold_idx}.pkl"), "wb") as f:
            pickle.dump(model, f)

    results_df = pd.DataFrame(results)
    results_df.to_csv(os.path.join(f"{outpath}/{mode}", f"{mode}_cv10_811_results.csv"), index=False)

    metric_cols = [
        "te_accuracy",
        "te_precision",
        "te_recall",
        "te_f1",
        "te_roc_auc",
        "te_mcc"
    ]

    summary_df = pd.DataFrame({
        "mean": results_df[metric_cols].mean(),
        "std": results_df[metric_cols].std(),
        "min": results_df[metric_cols].min(),
        "max": results_df[metric_cols].max()
    })

    summary_df.to_csv(os.path.join(f"{outpath}/{mode}",f"{mode}_cv10_811_summary.csv"))

    print(f"\n===== {mode.upper()} 10-Fold Summary =====")
    print(summary_df)

    return results_df, summary_df

In [ ]:
df = pd.read_csv(DATA_FILE)

X = df[X_COLS]
y = df[TARGET_COL]

In [ ]:
df = pd.read_csv(DATA_FILE)

for mode in ["xgb"]:   
    results_df, summary_df = run_model_cv(
        mode=mode,
        df=df,
        X_COLS=X_COLS,
        TARGET_COL=TARGET_COL,
        outpath=OUTPATH,
        pred_threshold=PRED_THRESHOLD,
        random_state=RANDOM_STATE
    )

XGB CV folds:  10%|█         | 1/10 [00:10<01:34, 10.52s/it]


===== XGB Fold 1 =====
Train / Val / Test = 1818 / 228 / 228
Test Recall: 0.9386
Test MCC:    0.9128
Test AUC:    0.9859


XGB CV folds:  20%|██        | 2/10 [00:22<01:31, 11.44s/it]


===== XGB Fold 2 =====
Train / Val / Test = 1818 / 228 / 228
Test Recall: 0.8947
Test MCC:    0.8806
Test AUC:    0.9760


XGB CV folds:  30%|███       | 3/10 [00:33<01:17, 11.11s/it]

...


===== XGB Fold 8 =====
Train / Val / Test = 1819 / 228 / 227
Test Recall: 0.8772
Test MCC:    0.8736
Test AUC:    0.9653


XGB CV folds:  90%|█████████ | 9/10 [01:50<00:12, 12.06s/it]


===== XGB Fold 9 =====
Train / Val / Test = 1819 / 228 / 227
Test Recall: 0.9123
Test MCC:    0.9060
Test AUC:    0.9833


XGB CV folds: 100%|██████████| 10/10 [02:02<00:00, 12.29s/it]


===== XGB Fold 10 =====
Train / Val / Test = 1819 / 228 / 227
Test Recall: 0.9035
Test MCC:    0.8789
Test AUC:    0.9848

===== XGB 10-Fold Summary =====
                  mean       std       min       max
te_accuracy   0.932709  0.015564  0.907489  0.956140
te_precision  0.971395  0.016670  0.944444  0.990476
te_recall     0.891771  0.024807  0.858407  0.938596
te_f1         0.929703  0.016827  0.902326  0.955357
te_roc_auc    0.975872  0.008245  0.964757  0.985919
te_mcc        0.868588  0.030561  0.818753  0.912843
